<a href="https://colab.research.google.com/github/latidore/Genesis/blob/main/inverse_trigonometric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Inverse Trigonometric Function Data Generation ---
# Create x values for arcsin and arccos (domain [-1, 1])
x_arcsin_arccos = np.linspace(-1, 1, 500)
y_arcsin = np.arcsin(x_arcsin_arccos)
y_arccos = np.arccos(x_arcsin_arccos)

# Create x values for arctan and arccot (domain all real numbers)
x_arctan_arccot = np.linspace(-10, 10, 500)
y_arctan = np.arctan(x_arctan_arccot)
y_arccot = np.pi / 2 - np.arctan(x_arctan_arccot) # arccot(x) = pi/2 - arctan(x)

# Create x values for arccsc and arcsec (domain (-inf, -1] U [1, inf))
# Need to handle the split domain carefully to avoid division by zero
x_arccsc_arcsec_neg = np.linspace(-10, -1.001, 250)
x_arccsc_arcsec_pos = np.linspace(1.001, 10, 250)

y_arccsc_neg = np.arcsin(1 / x_arccsc_arcsec_neg)
y_arccsc_pos = np.arcsin(1 / x_arccsc_arcsec_pos)

y_arcsec_neg = np.arccos(1 / x_arccsc_arcsec_neg)
y_arcsec_pos = np.arccos(1 / x_arccsc_arcsec_pos)

# Combine x and y for arccsc and arcsec to handle discontinuities with 'None'
x_arccsc_combined = np.concatenate((x_arccsc_arcsec_neg, [None], x_arccsc_arcsec_pos))
y_arccsc_combined = np.concatenate((y_arccsc_neg, [None], y_arccsc_pos))

x_arcsec_combined = np.concatenate((x_arccsc_arcsec_neg, [None], x_arccsc_arcsec_pos))
y_arcsec_combined = np.concatenate((y_arcsec_neg, [None], y_arcsec_pos))


# --- Trigonometric Function Data Generation ---
# 1. Define the x range for trigonometric functions
x_trig_start = -2.5 * np.pi
x_trig_end = 2.5 * np.pi

# 2. Generate data for sin(x) and cos(x)
x_sin_cos = np.linspace(x_trig_start, x_trig_end, 500)
y_sin = np.sin(x_sin_cos)
y_cos = np.cos(x_sin_cos)

# 3. Define parameters for handling discontinuities
num_points_per_segment = 100
epsilon = 0.01

# 4. Create a helper function named generate_discontinuous_function_data
def generate_discontinuous_function_data(func, discontinuities, x_start, x_end, n_points_segment, eps):
    x_combined = []
    y_combined = []

    # Filter and sort discontinuities within the given range
    relevant_discontinuities = sorted([d for d in discontinuities if x_start < d < x_end])

    current_segment_start = x_start

    for d in relevant_discontinuities:
        # Segment before discontinuity
        if d - eps > current_segment_start:
            x_segment = np.linspace(current_segment_start, d - eps, n_points_segment)
            y_segment = func(x_segment)
            x_combined.extend(x_segment)
            y_combined.extend(y_segment)

        # Add None for discontinuity
        x_combined.append(None)
        y_combined.append(None)

        # Update current_segment_start for the next segment
        current_segment_start = d + eps

    # Final segment after the last discontinuity or if no discontinuities
    if x_end > current_segment_start:
        x_segment = np.linspace(current_segment_start, x_end, n_points_segment)
        y_segment = func(x_segment)
        x_combined.extend(x_segment)
        y_combined.extend(y_segment)

    return np.array(x_combined, dtype=object), np.array(y_combined, dtype=object)

# 5. Generate data for tan(x) and sec(x)
tan_sec_discontinuities = np.array([-1.5, -0.5, 0.5, 1.5]) * np.pi

x_tan_combined, y_tan_combined = generate_discontinuous_function_data(
    np.tan, tan_sec_discontinuities, x_trig_start, x_trig_end, num_points_per_segment, epsilon
)

x_sec_combined, y_sec_combined = generate_discontinuous_function_data(
    lambda x: 1/np.cos(x), tan_sec_discontinuities, x_trig_start, x_trig_end, num_points_per_segment, epsilon
)

# 6. Generate data for cot(x) and csc(x)
cot_csc_discontinuities = np.array([-2, -1, 0, 1, 2]) * np.pi

x_cot_combined, y_cot_combined = generate_discontinuous_function_data(
    lambda x: 1/np.tan(x), cot_csc_discontinuities, x_trig_start, x_trig_end, num_points_per_segment, epsilon
)

x_csc_combined, y_csc_combined = generate_discontinuous_function_data(
    lambda x: 1/np.sin(x), cot_csc_discontinuities, x_trig_start, x_trig_end, num_points_per_segment, epsilon
)


# --- Create Subplot Layout ---
# Create a 1x2 subplot layout
fig = make_subplots(rows=1, cols=2, subplot_titles=('Inverse Trigonometric Functions', 'Trigonometric Functions'))


# --- Plot Inverse Trigonometric Functions ---
fig.add_trace(go.Scatter(x=x_arcsin_arccos, y=y_arcsin, mode='lines', name='arcsin(x)', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=x_arcsin_arccos, y=y_arccos, mode='lines', name='arccos(x)', line=dict(color='red')), row=1, col=1)
fig.add_trace(go.Scatter(x=x_arctan_arccot, y=y_arctan, mode='lines', name='arctan(x)', line=dict(color='green')), row=1, col=1)
fig.add_trace(go.Scatter(x=x_arctan_arccot, y=y_arccot, mode='lines', name='arccot(x)', line=dict(color='orange')), row=1, col=1)
fig.add_trace(go.Scatter(x=x_arccsc_combined, y=y_arccsc_combined, mode='lines', name='arccsc(x)', line=dict(color='purple')), row=1, col=1)
fig.add_trace(go.Scatter(x=x_arcsec_combined, y=y_arcsec_combined, mode='lines', name='arcsec(x)', line=dict(color='brown')), row=1, col=1)


# --- Plot Trigonometric Functions ---
fig.add_trace(go.Scatter(x=x_sin_cos, y=y_sin, mode='lines', name='sin(x)', line=dict(color='blue')), row=1, col=2)
fig.add_trace(go.Scatter(x=x_sin_cos, y=y_cos, mode='lines', name='cos(x)', line=dict(color='red')), row=1, col=2)
fig.add_trace(go.Scatter(x=x_tan_combined, y=y_tan_combined, mode='lines', name='tan(x)', line=dict(color='green')), row=1, col=2)
fig.add_trace(go.Scatter(x=x_cot_combined, y=y_cot_combined, mode='lines', name='cot(x)', line=dict(color='orange')), row=1, col=2)
fig.add_trace(go.Scatter(x=x_csc_combined, y=y_csc_combined, mode='lines', name='csc(x)', line=dict(color='purple')), row=1, col=2)
fig.add_trace(go.Scatter(x=x_sec_combined, y=y_sec_combined, mode='lines', name='sec(x)', line=dict(color='brown')), row=1, col=2)


# --- Update Layout and Annotations ---
fig.update_layout(
    title_text='Graphs of Inverse Trigonometric and Trigonometric Functions',
    hovermode='x unified',
    height=600,
    template='plotly_white',
    showlegend=True
)

# Update x and y axis labels for the left subplot (Inverse Trig Functions)
fig.update_xaxes(title_text='x', row=1, col=1,
                 tickvals=np.arange(-10, 11, 1),
                 ticktext=[str(i) for i in np.arange(-10, 11, 1)])
fig.update_yaxes(title_text='y', row=1, col=1,
                 tickvals=[-np.pi/2, 0, np.pi/2, np.pi],
                 ticktext=[r'$- \pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'],
                 range=[-np.pi/2 - 0.5, np.pi + 0.5])

# Update x and y axis labels for the right subplot (Trig Functions)
fig.update_xaxes(title_text='x', row=1, col=2,
                 tickvals=np.array([-2.5, -2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2, 2.5]) * np.pi,
                 ticktext=[r'$-5\pi/2$', r'$-2\pi$', r'$-3\pi/2$', r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$', r'$5\pi/2$'])
fig.update_yaxes(title_text='y', row=1, col=2, range=[-5, 5]) # Adjust range as needed for visibility

# Add horizontal lines to the left subplot (Inverse Trig Functions)
fig.add_hline(y=np.pi/2, line_dash='dash', line_color='gray', annotation_text=r'$\pi/2$', annotation_position='top right', row=1, col=1)
fig.add_hline(y=np.pi, line_dash='dash', line_color='gray', annotation_text=r'$\pi$', annotation_position='top right', row=1, col=1)
fig.add_hline(y=-np.pi/2, line_dash='dash', line_color='gray', annotation_text=r'$- \pi/2$', annotation_position='bottom right', row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', annotation_text=r'$0$', annotation_position='bottom left', row=1, col=1)

# Add horizontal lines to the right subplot (Trig Functions)
fig.add_hline(y=1, line_dash='dot', line_color='lightgray', annotation_text=r'$1$', annotation_position='top right', row=1, col=2)
fig.add_hline(y=-1, line_dash='dot', line_color='lightgray', annotation_text=r'$-1$', annotation_position='bottom right', row=1, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='gray', annotation_text=r'$0$', annotation_position='bottom left', row=1, col=2)


# --- Add y=x line to Both Subplots ---
# Data for the y=x line in the left subplot (Inverse Trig Functions)
x_yx_left = np.linspace(-3, 3, 100) # Choose a suitable range for y=x line on the left subplot
y_yx_left = x_yx_left

# Data for the y=x line in the right subplot (Trig Functions)
x_yx_right = np.linspace(-np.pi, np.pi, 100) # Choose a suitable range for y=x line on the right subplot
y_yx_right = x_yx_right

# Add y=x line to the left subplot (Inverse Trig Functions)
fig.add_trace(go.Scatter(x=x_yx_left, y=y_yx_left, mode='lines', name='y=x', line=dict(color='green', dash='dot')), row=1, col=1)

# Add y=x line to the right subplot (Trig Functions)
fig.add_trace(go.Scatter(x=x_yx_right, y=y_yx_right, mode='lines', name='y=x', line=dict(color='green', dash='dot')), row=1, col=2)


# --- Display Plot ---
fig.show()